# Visualización de Parches desde H5
### Explora los parches ya extraídos por `patch_extractor.py`

**Flujo:**
1. Configura la ruta a los archivos `.h5` en la Celda 2
2. Escanea los archivos disponibles en la Celda 3
3. Selecciona un archivo y visualiza parches aleatorios en la Celda 4
4. Explora estadísticas y coordenadas en las celdas siguientes

## Celda 1 — Imports

In [ ]:
import sys, os, warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import numpy as np
import h5py
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Rectangle
from PIL import Image
from tqdm.notebook import tqdm

Image.MAX_IMAGE_PIXELS = None

print("Librerías cargadas correctamente")

## Celda 2 — Configuración
**Edita la ruta base donde están los `.h5` de parches.**

In [ ]:
# ══════════════════════════════════════════════════════════════
# EDITA AQUÍ — Ruta a los archivos .h5 de parches
# ══════════════════════════════════════════════════════════════

H5_DIR = r"C:\ruta\a\tus\patches_h5"   # <-- CAMBIA ESTO

# Cantidad de parches a mostrar por grilla
PATCHES_PER_GRID = 16

# Semilla aleatoria para reproducibilidad
RANDOM_SEED = 42

# ══════════════════════════════════════════════════════════════

H5_DIR = Path(H5_DIR)
print(f"Directorio de búsqueda: {H5_DIR}")
print(f"Existe: {H5_DIR.exists()}")

np.random.seed(RANDOM_SEED)

## Celda 3 — Escanear archivos H5 disponibles
Lista todos los `.h5` en el directorio y muestra estadísticas básicas.

In [ ]:
if not H5_DIR.exists():
    print(f"ERROR: El directorio {H5_DIR} no existe.")
    print("Edita H5_DIR en la celda anterior.")
else:
    h5_files = sorted(H5_DIR.glob("*.h5"))
    # Excluir archivos de embeddings
    h5_files = [f for f in h5_files if not f.stem.endswith("_embeddings")]
    
    print(f"Archivos .h5 encontrados: {len(h5_files)}\n")
    
    if len(h5_files) == 0:
        print("No se encontraron archivos .h5 en ese directorio.")
    else:
        # Escanear metadata de cada archivo
        file_info = []
        for fpath in tqdm(h5_files, desc="Escaneando"):
            try:
                size_mb = fpath.stat().st_size / (1024 * 1024)
                with h5py.File(fpath, "r") as f:
                    n_patches = f["patches"].shape[0]
                    patch_size = f.attrs.get("patch_size", "?")
                    mag = f.attrs.get("magnification", "?")
                    slide = f.attrs.get("slide", fpath.stem)
                file_info.append({
                    "path": fpath,
                    "name": fpath.name,
                    "slide": slide,
                    "n_patches": n_patches,
                    "size_mb": size_mb,
                    "patch_size": patch_size,
                    "mag": mag,
                })
            except Exception as e:
                file_info.append({
                    "path": fpath,
                    "name": fpath.name,
                    "slide": "ERROR",
                    "n_patches": 0,
                    "size_mb": 0,
                    "patch_size": "?",
                    "mag": "?",
                    "error": str(e),
                })
        
        # Mostrar tabla resumen
        print(f"{'#':>3} {'Archivo':<45} {'Parches':>8} {'Tamaño':>8} {'Mag':>5}")
        print("─" * 75)
        for i, info in enumerate(file_info):
            err_flag = " ⚠" if "error" in info else ""
            print(f"{i+1:>3} {info['name']:<45} {info['n_patches']:>8,} {info['size_mb']:>7.1f}M {info['mag']!s:>5}{err_flag}")
        
        n_ok = sum(1 for info in file_info if "error" not in info)
        n_err = len(file_info) - n_ok
        total_patches = sum(info["n_patches"] for info in file_info)
        total_size = sum(info["size_mb"] for info in file_info)
        print(f"\nResumen: {n_ok} archivos OK, {n_err} con error")
        print(f"Total parches: {total_patches:,}")
        print(f"Total tamaño: {total_size:.1f} MB")

## Celda 4 — Seleccionar archivo y visualizar parches
Cambia el índice `SELECTED_IDX` para elegir qué archivo explorar.

In [ ]:
# ══════════════════════════════════════════════════════════════
# Selecciona el archivo por índice (1-based, según la tabla de arriba)
# ══════════════════════════════════════════════════════════════
SELECTED_IDX = 1   # <-- CAMBIA ESTE ÍNDICE
# ══════════════════════════════════════════════════════════════

if 'file_info' not in dir() or len(file_info) == 0:
    print("Primero ejecuta la Celda 3 para escanear archivos.")
elif SELECTED_IDX < 1 or SELECTED_IDX > len(file_info):
    print(f"Índice {SELECTED_IDX} fuera de rango (1-{len(file_info)}).")
else:
    sel = file_info[SELECTED_IDX - 1]
    print(f"Seleccionado: {sel['name']}")
    print(f"  Slide: {sel['slide']}")
    print(f"  Parches: {sel['n_patches']:,}")
    print(f"  Tamaño: {sel['size_mb']:.1f} MB")
    print(f"  Patch size: {sel['patch_size']}")
    print(f"  Magnificación: {sel['mag']}")
    
    if 'error' in sel:
        print(f"  ERROR: {sel['error']}")
    else:
        # Cargar datos
        f = h5py.File(sel['path'], "r")
        patches = f["patches"]
        coords = f["coords"]
        n_total = patches.shape[0]
        print(f"  Dataset patches shape: {patches.shape}")
        print(f"  Dataset coords shape: {coords.shape}")
        print(f"  dtype patches: {patches.dtype}")
        print(f"  dtype coords: {coords.dtype}")

## Celda 5 — Grilla de parches aleatorios
Muestra una grilla con parches aleatorios del archivo seleccionado.

In [ ]:
if 'patches' not in dir():
    print("Primero ejecuta la Celda 4 para cargar un archivo.")
else:
    n_total = patches.shape[0]
    n_show = min(PATCHES_PER_GRID, n_total)
    idxs = np.random.choice(n_total, size=n_show, replace=False)
    
    n_cols = int(np.ceil(np.sqrt(n_show)))
    n_rows = int(np.ceil(n_show / n_cols))
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows))
    fig.suptitle(f"Parches aleatorios — {sel['name']}", fontsize=14)
    axes = axes.flatten() if n_show > 1 else [axes]
    
    for i, idx in enumerate(idxs):
        patch = patches[idx]  # (C, H, W)
        if patch.shape[0] == 3:
            patch = np.transpose(patch, (1, 2, 0))  # (H, W, C)
        coord = coords[idx]
        
        axes[i].imshow(patch)
        axes[i].set_title(f"#{idx}  ({coord[0]}, {coord[1]})", fontsize=8)
        axes[i].axis("off")
    
    # Ocultar ejes sobrantes
    for i in range(n_show, len(axes)):
        axes[i].axis("off")
    
    plt.tight_layout()
    plt.show()
    
    print(f"Mostrando {n_show} parches de {n_total:,} totales")

## Celda 6 — Estadísticas de los parches
Distribución de intensidad/media de cada parche para detectar posibles problemas de calidad.

In [ ]:
if 'patches' not in dir():
    print("Primero ejecuta la Celda 4 para cargar un archivo.")
else:
    # Muestrear un subconjunto para estadísticas (evitar cargar todo en memoria)
    n_total = patches.shape[0]
    sample_size = min(5000, n_total)
    sample_idxs = np.random.choice(n_total, size=sample_size, replace=False)
    
    means = np.zeros(sample_size)
    stds = np.zeros(sample_size)
    zero_frac = np.zeros(sample_size)
    
    for i, idx in enumerate(tqdm(sample_idxs, desc="Calculando stats")):
        p = patches[idx]
        means[i] = p.mean()
        stds[i] = p.std()
        zero_frac[i] = (p == 0).mean()
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle("Estadísticas de parches (muestra de {:,})".format(sample_size), fontsize=13)
    
    axes[0].hist(means, bins=50, color="steelblue", edgecolor="white")
    axes[0].axvline(means.mean(), color="red", linestyle="--", label=f"media={means.mean():.1f}")
    axes[0].set_title("Media de intensidad")
    axes[0].set_xlabel("Valor medio (0-255)")
    axes[0].legend()
    
    axes[1].hist(stds, bins=50, color="seagreen", edgecolor="white")
    axes[1].set_title("Desviación estándar")
    axes[1].set_xlabel("Std")
    
    axes[2].hist(zero_frac, bins=50, color="coral", edgecolor="white")
    axes[2].set_title("Fracción de píxeles negros")
    axes[2].set_xlabel("Fracción == 0")
    
    plt.tight_layout()
    plt.show()
    
    # Detectar posibles parches problema
    problematic = zero_frac > 0.8
    if problematic.any():
        print(f"⚠ Se detectaron {problematic.sum()} parches con >80% píxeles negros "
              f"({problematic.mean()*100:.1f}% de la muestra).")

## Celda 7 — Distribución de coordenadas
Visualiza la posición espacial de los parches en la WSI original.

In [ ]:
if 'coords' not in dir():
    print("Primero ejecuta la Celda 4 para cargar un archivo.")
else:
    n_total = coords.shape[0]
    sample_size = min(10000, n_total)
    sample_idxs = np.random.choice(n_total, size=sample_size, replace=False)
    sample_coords = coords[sample_idxs]
    
    xs = sample_coords[:, 0]
    ys = sample_coords[:, 1]
    
    # Leer patch_size de atributos
    patch_size = f.attrs.get("patch_size", 224)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Scatter plot de posiciones
    ax.scatter(xs, ys, s=1, alpha=0.3, c="steelblue")
    ax.set_title(f"Distribución espacial de parches — {sel['name']} (muestra de {sample_size:,})", fontsize=12)
    ax.set_xlabel("Coordenada X")
    ax.set_ylabel("Coordenada Y")
    ax.set_aspect("equal")
    
    # Estadísticas
    x_min, x_max = xs.min(), xs.max()
    y_min, y_max = ys.min(), ys.max()
    ax.text(0.02, 0.98, 
            f"X: [{x_min}, {x_max}]\nY: [{y_min}, {y_max}]\n"
            f"Cobertura: {x_max - x_min + patch_size} x {y_max - y_min + patch_size} px\n"
            f"Parches totales: {n_total:,}",
            transform=ax.transAxes, va="top", fontsize=9,
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
    
    plt.tight_layout()
    plt.show()
    
    print(f"Rango X: {x_min} – {x_max} ({(x_max - x_min) / patch_size:.0f} parches)")
    print(f"Rango Y: {y_min} – {y_max} ({(y_max - y_min) / patch_size:.0f} parches)")

## Celda 8 — Inspección de parches con baja calidad
Busca y muestra parches con mucha área negra (posible padding) o muy oscuros.

In [ ]:
if 'patches' not in dir():
    print("Primero ejecuta la Celda 4 para cargar un archivo.")
else:
    n_total = patches.shape[0]
    
    # Umbrales de calidad
    BLACK_FRAC_THR = 0.9   # >90% negro -> muy probablemente padding
    MEAN_THR = 20          # media < 20 -> muy oscuro
    
    # Buscar secuencialmente (no cargar todo en memoria)
    bad_idxs = []
    scan_limit = min(5000, n_total)
    for idx in tqdm(range(scan_limit), desc="Buscando parches de baja calidad"):
        p = patches[idx]
        black_frac = (p == 0).mean()
        mean_val = p.mean()
        if black_frac > BLACK_FRAC_THR or mean_val < MEAN_THR:
            bad_idxs.append((idx, black_frac, mean_val))
    
    if len(bad_idxs) == 0:
        print(f"No se encontraron parches de baja calidad en los primeros {scan_limit} parches.")
    else:
        print(f"Se encontraron {len(bad_idxs)} parches de baja calidad en los primeros {scan_limit} parches.")
        
        # Mostrar hasta 12 ejemplos
        n_show = min(12, len(bad_idxs))
        n_cols = 4
        n_rows = int(np.ceil(n_show / n_cols))
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows))
        axes = axes.flatten()
        fig.suptitle("Parches de baja calidad detectados", fontsize=13)
        
        for i in range(n_show):
            idx, bf, mv = bad_idxs[i]
            p = patches[idx]
            if p.shape[0] == 3:
                p = np.transpose(p, (1, 2, 0))
            axes[i].imshow(p)
            axes[i].set_title(f"#{idx}  negro: {bf:.0%}", fontsize=8)
            axes[i].axis("off")
        
        for i in range(n_show, len(axes)):
            axes[i].axis("off")
        
        plt.tight_layout()
        plt.show()

## Celda 9 — Comparar múltiples archivos
Muestra una grilla con parches de varios archivos lado a lado.

In [ ]:
# ══════════════════════════════════════════════════════════════
# Índices de archivos a comparar (1-based)
# ══════════════════════════════════════════════════════════════
COMPARE_IDXS = [1, 2, 3]   # <-- CAMBIA ESTOS ÍNDICES
N_PER_FILE = 4              # parches por archivo
# ══════════════════════════════════════════════════════════════

if 'file_info' not in dir() or len(file_info) == 0:
    print("Primero ejecuta la Celda 3.")
else:
    valid_idxs = [i for i in COMPARE_IDXS if 1 <= i <= len(file_info)]
    if len(valid_idxs) == 0:
        print("Ningún índice válido.")
    else:
        n_files = len(valid_idxs)
        fig, axes = plt.subplots(N_PER_FILE, n_files, figsize=(3 * n_files, 3 * N_PER_FILE))
        if n_files == 1:
            axes = axes[:, None]
        
        for col, idx in enumerate(valid_idxs):
            info = file_info[idx - 1]
            try:
                with h5py.File(info["path"], "r") as f:
                    p = f["patches"]
                    c = f["coords"]
                    n_p = p.shape[0]
                    chosen = np.random.choice(n_p, size=min(N_PER_FILE, n_p), replace=False)
                    for row, pi in enumerate(chosen):
                        patch = p[pi]
                        if patch.shape[0] == 3:
                            patch = np.transpose(patch, (1, 2, 0))
                        axes[row, col].imshow(patch)
                        axes[row, col].axis("off")
                # Título de columna
                axes[0, col].set_title(f"{info['name'][:20]}\n{info['n_patches']:,}p", fontsize=8)
            except Exception as e:
                for row in range(N_PER_FILE):
                    axes[row, col].text(0.5, 0.5, f"ERROR\n{e}", ha="center", va="center", fontsize=7)
                    axes[row, col].axis("off")
        
        plt.tight_layout()
        plt.show()

## Celda 10 — Cerrar archivo H5 (limpieza)

In [ ]:
if 'f' in dir() and f:
    f.close()
    print("Archivo H5 cerrado.")